# Kaggle — Main paper: frozen GoogLeNet + PCA + RBF-SVM (16 and 64 patches)

Run this notebook **after** uploading the GitHub repository to Kaggle as an Input dataset (or after cloning your published GitHub repository into `/kaggle/working`) and attaching the original IQ-OTH/NCCD image dataset. Turn GPU on and enable Internet. This notebook does not use Random Forest. The entire calculation is in the Python scripts visible in `scripts/` and `src/`.

**Patient identifiers are unavailable: this is similarity-grouped image-level evaluation, NOT patient-disjoint evaluation.**


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

search_roots=[Path.cwd(),Path('/kaggle/working'),Path('/kaggle/input')]
repos=[]
for root in search_roots:
    if root.exists(): repos.extend(p.parent.parent for p in root.rglob('scripts/prepare_iqoth.py'))
if not repos:
    raise FileNotFoundError('Attach this GitHub repository as Kaggle Input, or copy/clone it into /kaggle/working.')
REPO=repos[0]
print('CODE REPOSITORY:',REPO)
print('GPU:',__import__('torch').cuda.is_available())
# Candidate source folders containing the exact original 3 classes.
folders=[p.parent for p in Path('/kaggle/input').rglob('Bengin cases') if p.is_dir()]
archives=[]
for p in Path('/kaggle/input').rglob('*.zip'):
    try:
        with zipfile.ZipFile(p) as z:
            names=z.namelist()
            if sum(n.lower().endswith('.jpg') and ('bengin cases' in n.lower() or 'malignant cases' in n.lower() or 'normal cases' in n.lower()) for n in names)>1000:
                archives.append(p)
    except zipfile.BadZipFile:pass
assert folders or archives, 'Add original IQ-OTH/NCCD dataset through Kaggle Add Input'
DATASET=str(archives[0] if archives else folders[0].parent)
print('ORIGINAL IQ-OTH INPUT:',DATASET)


In [ ]:
# Cell 2 — prepare exact 1054-image/432-group fold manifest.
import subprocess,sys,os
ENV=os.environ.copy(); ENV['PYTHONPATH']=str(REPO/'src')
WORK=Path('/kaggle/working/LUNG_CT_REPRO');WORK.mkdir(exist_ok=True)
subprocess.run([sys.executable,str(REPO/'scripts/prepare_iqoth.py'),'--dataset',DATASET,'--out',str(WORK/'prepared')],env=ENV,check=True)


In [ ]:
# Cell 3 — frozen GoogLeNet on GPU; 16 patches, 64 patches, and whole-image embeddings.
subprocess.run([sys.executable,str(REPO/'scripts/extract_googlenet_features.py'), '--dataset',DATASET,'--manifest',str(WORK/'prepared/manifest.csv'),'--grids','4','8','1','--out',str(WORK/'features')],env=ENV,check=True)


In [ ]:
# Cell 4 — all 17 fixed spatial candidates on leakage-safe group folds.
subprocess.run([sys.executable,str(REPO/'scripts/run_17_and_nested_cv.py'),'--mode','17','--cache',str(WORK/'features'),'--splits',str(WORK/'prepared'),'--out',str(WORK/'17_models')],env=ENV,check=True)
import pandas as pd
display(pd.read_csv(WORK/'17_models/ablation_17.csv'))


In [ ]:
# Cell 5 — nested 5x3 CV. May take several minutes; checkpoint files permit resume within same working directory.
subprocess.run([sys.executable,str(REPO/'scripts/run_17_and_nested_cv.py'),'--mode','nested','--cache',str(WORK/'features'),'--splits',str(WORK/'prepared'),'--out',str(WORK/'nested')],env=ENV,check=True)
display(pd.read_csv(WORK/'nested/nested_summary.csv'))


In [ ]:
# Cell 6 — save auditable outputs; large .npy arrays intentionally excluded.
import zipfile
out=Path('/kaggle/working/LUNG_CT_PAPER_RESULTS.zip')
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in [WORK/'prepared',WORK/'17_models',WORK/'nested']:
        for p in folder.rglob('*'):
            if p.is_file() and p.suffix in ('.csv','.json','.txt'):
                z.write(p,p.relative_to(WORK))
print('DOWNLOAD:',out)
